In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()

spark

In [7]:
df = spark.read.parquet("titanic.parquet")

In [8]:
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [21]:
df.select([
        (F.count(F.when(F.col(c).isNull(), c)) / df.count() * 100).alias(c)
        for c in df.columns
    ]).show()

+-----------+--------+------+----+---+------------------+-----+-----+------+----+-----------------+-------------------+
|PassengerId|Survived|Pclass|Name|Sex|               Age|SibSp|Parch|Ticket|Fare|            Cabin|           Embarked|
+-----------+--------+------+----+---+------------------+-----+-----+------+----+-----------------+-------------------+
|        0.0|     0.0|   0.0| 0.0|0.0|19.865319865319865|  0.0|  0.0|   0.0| 0.0|77.10437710437711|0.22446689113355783|
+-----------+--------+------+----+---+------------------+-----+-----+------+----+-----------------+-------------------+



In [24]:
df1 = df.fillna({"Age": df.select(F.mean(df['Age'])).collect()[0][0]})
df1.select([
        (F.count(F.when(F.col(c).isNull(), c)) / df1.count() * 100).alias(c)
        for c in df1.columns
    ]).show()

+-----------+--------+------+----+---+---+-----+-----+------+----+-----------------+-------------------+
|PassengerId|Survived|Pclass|Name|Sex|Age|SibSp|Parch|Ticket|Fare|            Cabin|           Embarked|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----------------+-------------------+
|        0.0|     0.0|   0.0| 0.0|0.0|0.0|  0.0|  0.0|   0.0| 0.0|77.10437710437711|0.22446689113355783|
+-----------+--------+------+----+---+---+-----+-----+------+----+-----------------+-------------------+



In [26]:
df.join(df1, on=["PassengerId","Pclass"], how="inner").show()

+-----------+------+--------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|PassengerId|Pclass|Survived|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Survived|                Name|   Sex|              Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+------+--------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|          1|     3|       0|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|       0|Braund, Mr. Owen ...|  male|             22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|     1|       1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       